In [ ]:
"""
Unified pipeline: Chile minerals domestic supply chain.

1. Download Sernageomin critical minerals from ArcGIS
2. Load USGS MINFAC_LAC (processing plants AND mines for ownership data)
3. Harmonize commodity names, fix naming inconsistencies
4. Parse secondary/by-product commodities into queryable form
5. Deduplicate overlapping records between sources
6. Run geographic consistency checks
7. Link mines to processing facilities by commodity + proximity
8. Load HS92 trade data and match to mineral facilities
9. Save outputs

Inputs:
  - ArcGIS FeatureServer (Sernageomin, downloaded live)
  - MINFAC_LAC.csv (USGS, local file)
  - Chile_HS92_Exports_6digit.csv (Harvard Growth Lab, local file)

Outputs:
  - Chile_Minerals_Inventory.csv / .geojson  (deduplicated facility list)
  - Chile_Mine_Plant_Links.csv               (mine-to-plant inferred links)
  - Chile_Supply_Chain_Summary.csv            (commodity-level domestic chain status)
"""

import requests
import os
import time
import warnings
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore", category=FutureWarning)

# ============================================================
# CONFIG
# ============================================================
USGS_PATH = "/Users/leoss/Downloads/MINFAC_LAC.csv"
HS92_PATH = "/Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile/Chile_HS92_Exports_6digit.csv"
OUTPUT_DIR = "/Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_URL = "https://services1.arcgis.com/OyjvVdFTl5hfSdX3/arcgis/rest/services/EMC_F/FeatureServer"

# Chile bounding box (mainland, excludes Easter Island)
CHILE_LAT_MIN, CHILE_LAT_MAX = -56.0, -17.5
CHILE_LON_MIN, CHILE_LON_MAX = -76.0, -66.0

# Max distance (km) to consider a mine-plant link plausible
LINK_MAX_KM = 150


# ============================================================
# COMMODITY HARMONIZATION
# ============================================================

# Spanish -> English + USGS inconsistencies
COMMODITY_MAP = {
    # Spanish to English
    "Cobre": "Copper", "Oro": "Gold", "Plata": "Silver",
    "Hierro": "Iron", "Litio": "Lithium", "Nitrato": "Nitrate",
    "Boro": "Boron", "Cinc": "Zinc", "Manganeso": "Manganese",
    "Titanio": "Titanium", "Tierras raras": "Rare Earths",
    "Molibdeno": "Molybdenum", "Cobalto": "Cobalt", "Renio": "Rhenium",
    "Potasio": "Potassium", "Yodo": "Iodine", "Selenio": "Selenium",
    "Telurio": "Tellurium", "Antimonio": "Antimony",
    # USGS English inconsistencies
    "Nitrates": "Nitrate",
    "Iron and steel": "Iron",
    "Sulfuric acid": "Sulfuric Acid",
    "Calcium carbonate": "Calcium Carbonate",
    "Phosphatic materials": "Phosphate",
    "Sodium sulfate": "Sodium Sulfate",
    "Potassium chloride": "Potassium",
    "Natural gas": "Natural Gas",
}


def harmonize_commodity(name):
    """Map a single commodity name to its harmonized English form."""
    if pd.isna(name):
        return name
    name = str(name).strip()
    return COMMODITY_MAP.get(name, name)


def parse_commodity_list(combo_string):
    """
    Parse a commodity combination string like 'Copper-Molybdenum-Rhenium'
    into a list of harmonized commodity names.
    """
    if pd.isna(combo_string):
        return []
    parts = str(combo_string).replace(",", "-").replace("/", "-").split("-")
    return [harmonize_commodity(p.strip()) for p in parts if p.strip()]


# ============================================================
# FACILITY TYPE CLASSIFICATION
# ============================================================

def classify_usgs_plant(name):
    """Classify USGS facility subtype from its name."""
    name = str(name).lower()
    if "smelter" in name:
        return "Smelter"
    if "refin" in name:
        return "Refinery"
    if "sx-ew" in name or "sx/ew" in name:
        return "SX-EW Plant"
    if "concentrat" in name or "flotation" in name or "milling" in name:
        return "Concentrator"
    if "pellet" in name:
        return "Pellet Plant"
    if "grinding" in name:
        return "Grinding Plant"
    if "steel" in name:
        return "Steel Plant"
    return "Processing Plant"


SERNA_STATUS_MAP = {
    "Mina en producción": "Active Mine",
    "Mina paralizada": "Idle Mine",
    "Mina Paralizada": "Idle Mine",
    "Prospecto/Proyecto": "Prospect/Project",
}

USGS_STATUS_MAP = {
    "A": "Active", "C": "Closed", "S": "Standby", "CM": "Care & Maintenance",
}

# Stage in value chain, used for mine-to-plant linking
FACILITY_STAGE = {
    "Mine (active)": "extraction",
    "Mine (idle)": "extraction",
    "Prospect/Project": "extraction",
    "Concentrator": "processing",
    "SX-EW Plant": "processing",
    "Smelter": "processing",
    "Refinery": "processing",
    "Processing Plant": "processing",
    "Pellet Plant": "processing",
    "Grinding Plant": "processing",
    "Steel Plant": "processing",
}


# ============================================================
# HELPER: ArcGIS download
# ============================================================

def download_layer(layer_id, with_geometry=True):
    """Download all records from a Sernageomin ArcGIS FeatureServer layer."""
    all_features = []
    offset = 0
    while True:
        params = {
            "where": "1=1", "outFields": "*", "f": "json",
            "resultRecordCount": 2000, "resultOffset": offset,
        }
        if with_geometry:
            params["outSR"] = "4326"
            params["returnGeometry"] = "true"
        r = requests.get(f"{BASE_URL}/{layer_id}/query", params=params, timeout=60)
        r.raise_for_status()
        data = r.json()
        features = data.get("features", [])
        if not features:
            break
        all_features.extend(features)
        if len(features) < 2000:
            break
        offset += len(features)
    return all_features


# ============================================================
# HELPER: Haversine distance
# ============================================================

def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorized haversine distance in km."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))


# ============================================================
# STEP 1: Download Sernageomin
# ============================================================

def step1_sernageomin():
    print("=" * 60)
    print("STEP 1: Downloading Sernageomin critical minerals...")
    print("=" * 60)

    dep_features = download_layer(0, with_geometry=True)
    print(f"  Deposits: {len(dep_features)}")

    res_features = download_layer(3, with_geometry=False)
    print(f"  Reserve/resource records: {len(res_features)}")

    # Build GeoDataFrame from deposits
    rows = []
    for feat in dep_features:
        row = feat.get("attributes", {})
        geom = feat.get("geometry")
        if geom and "x" in geom:
            row["geometry"] = Point(geom["x"], geom["y"])
            row["LATITUD"] = geom["y"]
            row["LONGITUD"] = geom["x"]
        else:
            row["geometry"] = None
        rows.append(row)
    serna_gdf = gpd.GeoDataFrame(pd.DataFrame(rows), geometry="geometry", crs="EPSG:4326")

    # Pivot reserves into one row per deposit
    reservas = pd.DataFrame([f["attributes"] for f in res_features])
    if len(reservas) > 0:
        reservas_pivot = reservas.pivot_table(
            index="ID_DEPOSITO", columns=["MINERAL", "TIPO"],
            values="VALOR", aggfunc="sum"
        )
        reservas_pivot.columns = [f"{mineral}_{tipo}" for mineral, tipo in reservas_pivot.columns]
        reservas_pivot = reservas_pivot.reset_index()
        serna_gdf = serna_gdf.merge(reservas_pivot, left_on="ID", right_on="ID_DEPOSITO", how="left")
        serna_gdf = serna_gdf.drop(columns=["ID_DEPOSITO"], errors="ignore")

    # Standardize columns
    serna_gdf["SOURCE"] = "Sernageomin_2025"
    serna_gdf["STATUS"] = serna_gdf["ESTADO_DEPOSITO"].map({
    "Mina en producción": "Active",
    "Mina paralizada": "Idle",
    "Mina Paralizada": "Idle",
    "Prospecto/Proyecto": "Prospect/Project",
    })
    serna_gdf["FACILITY_TYPE"] = serna_gdf["ESTADO_DEPOSITO"].map({
    "Mina en producción": "Mine (active)",
    "Mina paralizada": "Mine (idle)",
    "Mina Paralizada": "Mine (idle)",
    "Prospecto/Proyecto": "Prospect/Project",
})
    serna_gdf["FACILITY_NAME"] = serna_gdf["NOMBRE_DEPOSITO"]
    serna_gdf["PRIMARY_COMMODITY"] = serna_gdf["CRITICO_1"].apply(harmonize_commodity)
    serna_gdf["ALL_COMMODITIES_RAW"] = serna_gdf["COMBINACION_CRITICO"]
    serna_gdf["OPERATOR_NAME"] = None
    serna_gdf["OWNER_NAME"] = None
    serna_gdf["CAPACITY"] = None
    serna_gdf["CAPACITY_UNITS"] = None


    # Rename reserve columns to English
    rename_cols = {}
    for col in serna_gdf.columns:
        new_col = col
        for es, en in COMMODITY_MAP.items():
            if es in new_col:
                new_col = new_col.replace(es, en)
        new_col = new_col.replace("_Recurso", "_Resource").replace("_Reserva", "_Reserve")
        if new_col != col:
            rename_cols[col] = new_col
    serna_gdf = serna_gdf.rename(columns=rename_cols)

    print(f"  Processed: {serna_gdf.shape}")
    return serna_gdf


# ============================================================
# STEP 2: Load USGS (processing plants AND mines for ownership)
# ============================================================

def step2_usgs():
    print("\n" + "=" * 60)
    print("STEP 2: Loading USGS mineral facilities...")
    print("=" * 60)

    usgs = pd.read_csv(USGS_PATH, encoding="latin-1")
    chile_usgs = usgs[usgs["COUNTRY"].str.strip().str.lower() == "chile"].copy()
    print(f"  USGS Chile total: {len(chile_usgs)}")

    # Processing facilities (same filter as before)
    processing_codes = ["P", "B", "OR", "MP", "p"]
    plants = chile_usgs[chile_usgs["FACTYPE"].isin(processing_codes)].copy()
    plants["FACILITY_TYPE"] = plants["LOCNAME"].apply(classify_usgs_plant)
    print(f"  Processing facilities: {len(plants)}")

    # Mines from USGS (for ownership/operator data, will be used in dedup matching)
    mine_codes = ["M", "m", "DM", "SM"]
    usgs_mines = chile_usgs[chile_usgs["FACTYPE"].isin(mine_codes)].copy()
    usgs_mines["FACILITY_TYPE"] = "Mine (USGS)"
    print(f"  USGS mine records (for ownership matching): {len(usgs_mines)}")

    # Build GeoDataFrame for plants
    plants["geometry"] = plants.apply(
        lambda r: Point(r["DDLONG"], r["DDLAT"])
        if pd.notna(r["DDLAT"]) and pd.notna(r["DDLONG"]) else None,
        axis=1
    )
    plants_gdf = gpd.GeoDataFrame(plants, geometry="geometry", crs="EPSG:4326")

    # Standardize columns
    plants_gdf["SOURCE"] = "USGS_MINFAC_2017"
    plants_gdf["FACILITY_NAME"] = plants_gdf["LOCNAME"]
    plants_gdf["PRIMARY_COMMODITY"] = plants_gdf["COMMODITY"].apply(harmonize_commodity)
    plants_gdf["ALL_COMMODITIES_RAW"] = plants_gdf["COMMODITY"]
    plants_gdf["OPERATOR_NAME"] = plants_gdf["OPERATOR"]
    plants_gdf["OWNER_NAME"] = plants_gdf["OWNER"]
    plants_gdf["CAPACITY"] = plants_gdf["ANNCAP"]
    plants_gdf["CAPACITY_UNITS"] = plants_gdf["UNITS"]
    plants_gdf["LATITUD"] = plants_gdf["DDLAT"]
    plants_gdf["LONGITUD"] = plants_gdf["DDLONG"]
    plants_gdf["REGION"] = plants_gdf["ADM1"]
    plants_gdf["STATUS"] = plants_gdf["STATUS"].map(USGS_STATUS_MAP).fillna(plants_gdf["STATUS"])

    # Keep USGS mines as a separate df for ownership enrichment
    usgs_mines_df = usgs_mines[["LOCNAME", "COMMODITY", "OPERATOR", "OWNER",
                                 "DDLAT", "DDLONG"]].copy()
    usgs_mines_df["LOCNAME_lower"] = usgs_mines_df["LOCNAME"].str.lower().str.strip()

    print(f"  Plants processed: {plants_gdf.shape}")
    return plants_gdf, usgs_mines_df


# ============================================================
# STEP 3: Merge and deduplicate
# ============================================================

def step3_merge_and_dedup(serna_gdf, plants_gdf, usgs_mines_df):
    print("\n" + "=" * 60)
    print("STEP 3: Merging and deduplicating...")
    print("=" * 60)

    # Identify common columns, carry everything
    # Build a union of columns, fill missing with None
    all_cols = list(set(serna_gdf.columns) | set(plants_gdf.columns))
    for col in all_cols:
        if col not in serna_gdf.columns:
            serna_gdf[col] = None
        if col not in plants_gdf.columns:
            plants_gdf[col] = None

    combined = gpd.GeoDataFrame(
        pd.concat([serna_gdf[all_cols], plants_gdf[all_cols]], ignore_index=True),
        crs="EPSG:4326"
    )
    print(f"  Before dedup: {len(combined)} records")

    # --- Deduplication ---
    # Strategy: for each Sernageomin record, check if a USGS record exists
    # with a similar name AND same primary commodity AND within 10km.
    # If so, enrich the Sernageomin record with USGS ownership data and drop the USGS duplicate.
    #
    # This primarily catches mine/extraction overlaps. Processing plants from USGS
    # and mines from Sernageomin are different facility types and should coexist.

    serna_mask = combined["SOURCE"] == "Sernageomin_2025"
    usgs_mask = combined["SOURCE"] == "USGS_MINFAC_2017"

    # Also try to enrich Sernageomin ownership from USGS mine records
    serna_rows = combined[serna_mask].copy()
    serna_rows["name_lower"] = serna_rows["FACILITY_NAME"].str.lower().str.strip()

    enriched_count = 0
    for idx, row in serna_rows.iterrows():
        # Fuzzy match: check if USGS mines have a record with partial name overlap
        matches = usgs_mines_df[
            usgs_mines_df["LOCNAME_lower"].str.contains(
                row["name_lower"][:8], case=False, na=False, regex=False
            )
        ]
        if len(matches) > 0:
            # Take the first match's ownership info
            best = matches.iloc[0]
            if pd.isna(combined.at[idx, "OPERATOR_NAME"]) and pd.notna(best["OPERATOR"]):
                combined.at[idx, "OPERATOR_NAME"] = best["OPERATOR"]
                enriched_count += 1
            if pd.isna(combined.at[idx, "OWNER_NAME"]) and pd.notna(best["OWNER"]):
                combined.at[idx, "OWNER_NAME"] = best["OWNER"]

    print(f"  Enriched {enriched_count} Sernageomin records with USGS ownership data")

    # Flag exact name duplicates (same name in both sources, same commodity, both extraction)
    combined["_name_lower"] = combined["FACILITY_NAME"].str.lower().str.strip()
    combined["_stage"] = combined["FACILITY_TYPE"].map(FACILITY_STAGE)

    dupes_to_drop = []
    serna_extract = combined[serna_mask & (combined["_stage"] == "extraction")]
    usgs_rows = combined[usgs_mask]

    for _, srow in serna_extract.iterrows():
        # Check USGS for same-name, same-commodity facility
        potential = usgs_rows[
            (usgs_rows["_name_lower"] == srow["_name_lower"]) &
            (usgs_rows["PRIMARY_COMMODITY"] == srow["PRIMARY_COMMODITY"])
        ]
        if len(potential) > 0:
            dupes_to_drop.extend(potential.index.tolist())

    dupes_to_drop = list(set(dupes_to_drop))
    if dupes_to_drop:
        combined = combined.drop(index=dupes_to_drop)
        print(f"  Dropped {len(dupes_to_drop)} USGS duplicates of Sernageomin records")

    combined = combined.drop(columns=["_name_lower", "_stage"], errors="ignore")
    combined = combined.reset_index(drop=True)
    print(f"  After dedup: {len(combined)} records")

    return combined


# ============================================================
# STEP 4: Parse secondary commodities
# ============================================================

def step4_parse_commodities(combined):
    print("\n" + "=" * 60)
    print("STEP 4: Parsing secondary commodities...")
    print("=" * 60)

    # Parse ALL_COMMODITIES_RAW into a list column
    combined["COMMODITY_LIST"] = combined["ALL_COMMODITIES_RAW"].apply(parse_commodity_list)

    # Also create a flat lookup: facility_index -> list of commodities
    # This is used in Step 7 for mine-plant linking on secondary commodities
    combined["N_COMMODITIES"] = combined["COMMODITY_LIST"].apply(len)

    multi = combined[combined["N_COMMODITIES"] > 1]
    print(f"  Facilities with multiple commodities: {len(multi)}")
    print(f"  Max commodities per facility: {combined['N_COMMODITIES'].max()}")
    
    exploded = combined.explode("COMMODITY_LIST").reset_index(drop=True)
    exploded = exploded.drop(columns=["COMMODITY"], errors="ignore")
    exploded = exploded.rename(columns={"COMMODITY_LIST": "COMMODITY"})
    mask = exploded["COMMODITY"].notna() & (exploded["COMMODITY"].astype(str) != "")
    exploded = exploded.loc[mask.values].reset_index(drop=True) 

    commodity_counts = exploded.groupby("COMMODITY")["FACILITY_NAME"].count().sort_values(ascending=False)
    print(f"\n  Unique commodities after parsing: {len(commodity_counts)}")
    print(f"  Top 15:")
    for c, n in commodity_counts.head(15).items():
        print(f"    {c}: {n} facilities")

    return combined, exploded


# ============================================================
# STEP 5: Geographic consistency checks
# ============================================================

def step5_geo_checks(combined):
    print("\n" + "=" * 60)
    print("STEP 5: Geographic consistency checks...")
    print("=" * 60)

    has_coords = combined[combined["LATITUD"].notna() & combined["LONGITUD"].notna()].copy()
    print(f"  Records with coordinates: {len(has_coords)} / {len(combined)}")

    outside = has_coords[
        (has_coords["LONGITUD"] > CHILE_LON_MAX) |
        (has_coords["LONGITUD"] < CHILE_LON_MIN) |
        (has_coords["LATITUD"] > CHILE_LAT_MAX) |
        (has_coords["LATITUD"] < CHILE_LAT_MIN)
    ]

    if len(outside) > 0:
        print(f"\n  *** {len(outside)} records OUTSIDE Chile bounding box ***")
        print(outside[["FACILITY_NAME", "REGION", "LONGITUD", "LATITUD",
                        "PRIMARY_COMMODITY", "SOURCE"]].to_string(index=False))
    else:
        print("  All records within Chile bounding box.")

    # Border zone (near Argentine border)
    border = has_coords[has_coords["LONGITUD"] > -67.5].sort_values("LONGITUD", ascending=False)
    print(f"\n  Border-zone records (lon > -67.5): {len(border)}")
    if len(border) > 0:
        print(border[["FACILITY_NAME", "REGION", "LONGITUD", "LATITUD",
                       "PRIMARY_COMMODITY"]].to_string(index=False))

    return combined


# ============================================================
# STEP 6: Link mines to processing facilities
# ============================================================

def step6_mine_plant_links(combined, exploded):
    print("\n" + "=" * 60)
    print("STEP 6: Linking mines to processing facilities...")
    print("=" * 60)

    combined["_stage"] = combined["FACILITY_TYPE"].map(FACILITY_STAGE)

    mines = combined[
        (combined["_stage"] == "extraction") &
        combined["LATITUD"].notna() &
        combined["LONGITUD"].notna()
    ].copy()

    plants = combined[
        (combined["_stage"] == "processing") &
        combined["LATITUD"].notna() &
        combined["LONGITUD"].notna()
    ].copy()

    print(f"  Mines/prospects with coords: {len(mines)}")
    print(f"  Processing facilities with coords: {len(plants)}")

    if len(mines) == 0 or len(plants) == 0:
        print("  Cannot build links: missing mines or plants.")
        return pd.DataFrame()

    # For each mine, find all plants that process any of its commodities
    # within LINK_MAX_KM, ranked by distance.
    # Uses the exploded commodity lists so by-products are matched.

    # Build commodity sets for each facility
    mine_commodities = {}
    for idx, row in mines.iterrows():
        mine_commodities[idx] = set(row.get("COMMODITY_LIST", []) or [row["PRIMARY_COMMODITY"]])

    plant_commodities = {}
    for idx, row in plants.iterrows():
        plant_commodities[idx] = set(row.get("COMMODITY_LIST", []) or [row["PRIMARY_COMMODITY"]])

    links = []
    for midx, mrow in mines.iterrows():
        m_comms = mine_commodities[midx]
        if not m_comms:
            continue

        for pidx, prow in plants.iterrows():
            p_comms = plant_commodities[pidx]
            shared = m_comms & p_comms
            if not shared:
                continue

            dist = haversine_km(
                mrow["LATITUD"], mrow["LONGITUD"],
                prow["LATITUD"], prow["LONGITUD"]
            )

            if dist <= LINK_MAX_KM:
                links.append({
                    "MINE_NAME": mrow["FACILITY_NAME"],
                    "MINE_IDX": midx,
                    "MINE_TYPE": mrow["FACILITY_TYPE"],
                    "MINE_STATUS": mrow["STATUS"],
                    "MINE_LAT": mrow["LATITUD"],
                    "MINE_LON": mrow["LONGITUD"],
                    "MINE_REGION": mrow.get("REGION"),
                    "MINE_OPERATOR": mrow.get("OPERATOR_NAME"),
                    "PLANT_NAME": prow["FACILITY_NAME"],
                    "PLANT_IDX": pidx,
                    "PLANT_TYPE": prow["FACILITY_TYPE"],
                    "PLANT_STATUS": prow["STATUS"],
                    "PLANT_LAT": prow["LATITUD"],
                    "PLANT_LON": prow["LONGITUD"],
                    "PLANT_OPERATOR": prow.get("OPERATOR_NAME"),
                    "PLANT_OWNER": prow.get("OWNER_NAME"),
                    "PLANT_CAPACITY": prow.get("CAPACITY"),
                    "PLANT_CAPACITY_UNITS": prow.get("CAPACITY_UNITS"),
                    "SHARED_COMMODITIES": ", ".join(sorted(shared)),
                    "DISTANCE_KM": round(dist, 1),
                })

    links_df = pd.DataFrame(links)

    if len(links_df) > 0:
        links_df = links_df.sort_values(["MINE_NAME", "DISTANCE_KM"])
        print(f"  Total mine-plant links found: {len(links_df)}")
        print(f"  Unique mines with at least one link: {links_df['MINE_NAME'].nunique()}")
        print(f"  Unique plants linked: {links_df['PLANT_NAME'].nunique()}")

        # Mines with NO link
        linked_mine_idxs = set(links_df["MINE_IDX"])
        unlinked = mines[~mines.index.isin(linked_mine_idxs)]
        print(f"\n  Mines/prospects with NO nearby processing: {len(unlinked)}")
        if len(unlinked) > 0:
            print(f"  By commodity:")
            for c, n in unlinked["PRIMARY_COMMODITY"].value_counts().items():
                print(f"    {c}: {n}")
    else:
        print("  No links found within distance threshold.")

    combined = combined.drop(columns=["_stage"], errors="ignore")
    return links_df


# ============================================================
# STEP 7: Cross-reference with HS92 trade data
# ============================================================

# Mapping from mineral commodity names to HS92 product codes (6-digit)
# These are approximate: one mineral can map to multiple HS codes
HS92_COMMODITY_MAP = {
    "Copper": {
        "260300": "Copper ores and concentrates",
        "740311": "Refined copper, cathodes",
        "740312": "Refined copper, wire bars",
        "740400": "Copper waste and scrap",
        "740319": "Refined copper, other",
        "740321": "Copper alloys, copper-zinc",
        "740329": "Copper alloys, other",
    },
    "Lithium": {
        "283691": "Lithium carbonate",
        "253090": "Lithium minerals (ores)",
    },
    "Molybdenum": {
        "261310": "Molybdenum ores, roasted",
        "261390": "Molybdenum ores, other",
        "810294": "Molybdenum, unwrought",
    },
    "Gold": {
        "710812": "Gold, non-monetary, unwrought",
        "261690": "Precious metal ores",
    },
    "Silver": {
        "710691": "Silver, unwrought",
        "261610": "Silver ores and concentrates",
    },
    "Iron": {
        "260111": "Iron ores, non-agglomerated",
        "260112": "Iron ores, agglomerated",
        "720110": "Pig iron",
        "720241": "Ferrochromium",
    },
    "Nitrate": {
        "310239": "Sodium nitrate",
        "310250": "Sodium nitrate, natural",
        "283410": "Nitrites",
    },
    "Iodine": {
        "280120": "Iodine",
    },
    "Boron": {
        "252810": "Borates, natural",
        "281000": "Boric acid",
    },
    "Zinc": {
        "260800": "Zinc ores and concentrates",
        "790111": "Zinc, not alloyed, unwrought",
    },
    "Rhenium": {
        "811292": "Rhenium, unwrought",
    },
    "Potassium": {
        "310420": "Potassium chloride",
    },
    "Rare Earths": {
        "280530": "Rare earth metals",
    },
    "Selenium": {
        "280490": "Selenium",
    },
}


def step7_trade_crossref(combined, links_df):
    print("\n" + "=" * 60)
    print("STEP 7: Cross-referencing with HS92 trade data...")
    print("=" * 60)

    if not os.path.exists(HS92_PATH):
        print(f"  HS92 file not found at {HS92_PATH}, skipping trade analysis.")
        return None

    trade = pd.read_csv(HS92_PATH)
    print(f"  Trade data loaded: {trade.shape}")

    # Detect column names
    cols = trade.columns.tolist()
    product_col = "product_hs92_code"
    export_col = next((c for c in cols if c == "export_value"), None)
    year_col = next((c for c in cols if c == "year"), None)
    export_col = next((c for c in cols if "export" in c.lower()), None)
    year_col = next((c for c in cols if "year" in c.lower()), None)

    if not all([product_col, export_col, year_col]):
        print(f"  Could not detect columns. Available: {cols}")
        return None

    print(f"  Using columns: product={product_col}, export={export_col}, year={year_col}")

    # Ensure product codes are strings with leading zeros
    trade[product_col] = trade[product_col].astype(str).str.zfill(6)

    # Get most recent year
    latest_year = trade[year_col].max()
    print(f"  Latest year in data: {latest_year}")

    # For each mineral commodity, look up its HS92 codes and sum exports
    summary_rows = []

    for commodity, hs_codes in HS92_COMMODITY_MAP.items():
        # Count domestic facilities
        n_mines = len(combined[
            (combined["FACILITY_TYPE"].isin(["Mine (active)", "Mine (idle)", "Prospect/Project"])) &
            (combined["COMMODITY_LIST"].apply(lambda x: commodity in (x or [])))
        ])
        n_active_mines = len(combined[
            (combined["FACILITY_TYPE"] == "Mine (active)") &
            (combined["COMMODITY_LIST"].apply(lambda x: commodity in (x or [])))
        ])
        n_plants = len(combined[
            (combined["FACILITY_TYPE"].map(FACILITY_STAGE) == "processing") &
            (combined["COMMODITY_LIST"].apply(lambda x: commodity in (x or [])))
        ])

        # Count mine-plant links
        if len(links_df) > 0:
            n_links = len(links_df[links_df["SHARED_COMMODITIES"].str.contains(commodity, na=False)])
        else:
            n_links = 0

        # Sum trade values for latest year
        hs_list = list(hs_codes.keys())
        latest_trade = trade[(trade[year_col] == latest_year) & (trade[product_col].isin(hs_list))]
        total_export = latest_trade[export_col].sum() if len(latest_trade) > 0 else 0

        # Distinguish raw vs processed exports where possible
        # Convention: ores/concentrates are "raw", metals/refined are "processed"
        raw_codes = [c for c in hs_list if any(k in hs_codes[c].lower()
                     for k in ["ore", "concentrate", "mineral", "natural"])]
        proc_codes = [c for c in hs_list if c not in raw_codes]

        raw_export = trade[
            (trade[year_col] == latest_year) & (trade[product_col].isin(raw_codes))
        ][export_col].sum() if raw_codes else 0

        proc_export = trade[
            (trade[year_col] == latest_year) & (trade[product_col].isin(proc_codes))
        ][export_col].sum() if proc_codes else 0

        # Determine chain status
        if n_active_mines > 0 and n_plants > 0:
            chain_status = "Extraction + Processing"
        elif n_active_mines > 0 and n_plants == 0:
            chain_status = "Extraction only"
        elif n_mines == 0 and n_plants > 0:
            chain_status = "Processing only (no extraction in dataset)"
        else:
            chain_status = "No active facilities"

        summary_rows.append({
            "COMMODITY": commodity,
            "ACTIVE_MINES": n_active_mines,
            "TOTAL_MINES_PROSPECTS": n_mines,
            "PROCESSING_FACILITIES": n_plants,
            "MINE_PLANT_LINKS": n_links,
            "CHAIN_STATUS": chain_status,
            f"EXPORT_TOTAL_{latest_year}_USD": total_export,
            f"EXPORT_RAW_{latest_year}_USD": raw_export,
            f"EXPORT_PROCESSED_{latest_year}_USD": proc_export,
            "RAW_SHARE": round(raw_export / total_export, 3) if total_export > 0 else None,
        })

    summary = pd.DataFrame(summary_rows)
    summary = summary.sort_values(f"EXPORT_TOTAL_{latest_year}_USD", ascending=False)

    print(f"\n  Supply chain summary ({latest_year}):")
    print(summary.to_string(index=False))

    return summary


# ============================================================
# STEP 8: Save outputs
# ============================================================

def step8_save(combined, links_df, summary):
    print("\n" + "=" * 60)
    print("STEP 8: Saving outputs...")
    print("=" * 60)

    # Inventory (drop internal helper columns)
    drop_cols = ["_stage", "_name_lower", "COMMODITY_LIST"]
    out = combined.drop(columns=[c for c in drop_cols if c in combined.columns], errors="ignore")

    # GeoJSON
    geo_path = os.path.join(OUTPUT_DIR, "Chile_Minerals_Inventory.geojson")
    # COMMODITY_LIST is a list, which geojson can't serialize; convert to string
    if "COMMODITY_LIST" in out.columns:
        out["COMMODITY_LIST"] = out["COMMODITY_LIST"].apply(
            lambda x: ", ".join(x) if isinstance(x, list) else x
        )
    out.to_file(geo_path, driver="GeoJSON")
    print(f"  Saved: {geo_path}")

    # CSV (drop geometry)
    csv_path = os.path.join(OUTPUT_DIR, "Chile_Minerals_Inventory.csv")
    pd.DataFrame(out.drop(columns="geometry")).to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path}")

    # Mine-plant links
    if len(links_df) > 0:
        links_path = os.path.join(OUTPUT_DIR, "Chile_Mine_Plant_Links.csv")
        links_df.to_csv(links_path, index=False)
        print(f"  Saved: {links_path}")

    # Supply chain summary
    if summary is not None:
        summary_path = os.path.join(OUTPUT_DIR, "Chile_Supply_Chain_Summary.csv")
        summary.to_csv(summary_path, index=False)
        print(f"  Saved: {summary_path}")

    print(f"\n  Total inventory records: {len(out)}")


# ============================================================
# MAIN
# ============================================================

def main():
    start = time.time()

    serna_gdf = step1_sernageomin()
    plants_gdf, usgs_mines_df = step2_usgs()
    combined = step3_merge_and_dedup(serna_gdf, plants_gdf, usgs_mines_df)
    combined, exploded = step4_parse_commodities(combined)
    combined = step5_geo_checks(combined)
    links_df = step6_mine_plant_links(combined, exploded)
    summary = step7_trade_crossref(combined, links_df)
    step8_save(combined, links_df, summary)

    elapsed = time.time() - start
    print(f"\nPipeline completed in {elapsed:.1f}s")


if __name__ == "__main__":
    main()

Loading mine-plant link data...
  Links: 1370
  Unique mines: 206
  Unique plants: 78
  Inventory loaded: 461 facilities

Building bipartite network...
  Nodes: 284 (206 mines, 78 plants)
  Edges: 1272 unique mine-plant pairs
  Density: 0.0317
  Connected components: 8
  Largest component: 248 nodes (87.3%)

Computing centrality metrics...

  Top 10 mines by degree:
    Penacho Blanco: deg=20, betw=0.0529
    Polo Sur: deg=19, betw=0.0202
    Centinela: deg=19, betw=0.0001
    Mirador: deg=19, betw=0.0839
    Chuquicamata: deg=16, betw=0.0351
    Sierra Gorda: deg=16, betw=0.0001
    Lomas Bayas: deg=16, betw=0.0074
    Radomiro Tomic: deg=16, betw=0.0022
    El Abra: deg=15, betw=0.0213
    Antucoya: deg=15, betw=0.0009

  Top 10 plants by degree:
    Centinela oxide and sulfide plant: deg=39, betw=0.0116
    El Tesoro SX-EW plant: deg=39, betw=0.0010
    Spence SX-EW plant: deg=34, betw=0.0004
    Lomas Bayas SX-EW plant: deg=33, betw=0.0047
    Antucoya SX-EW plant: deg=33, betw=0.0

In [78]:
# ============================================================
# ADDITIONAL VISUALIZATIONS
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "figure.facecolor": "white",
    "axes.facecolor": "#fafafa",
    "axes.edgecolor": "#cccccc",
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.5,
})

def plot_centrality_distributions(metrics, output_dir):
    """Degree, betweenness, eigenvector distributions split by node type."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))

    measures = [
        ("DEGREE", "Degree"),
        ("BETWEENNESS_CENTRALITY", "Betweenness Centrality"),
        ("EIGENVECTOR_CENTRALITY", "Eigenvector Centrality"),
    ]

    mine_m = metrics[metrics["NODE_TYPE"] == "mine"]
    plant_m = metrics[metrics["NODE_TYPE"] == "plant"]

    for col_idx, (col, label) in enumerate(measures):
        # Top row: histograms
        ax = axes[0, col_idx]
        bins = np.linspace(0, metrics[col].quantile(0.98), 25)
        ax.hist(mine_m[col], bins=bins, alpha=0.7, color="#4dabf7", label="Mines", edgecolor="white", linewidth=0.4)
        ax.hist(plant_m[col], bins=bins, alpha=0.7, color="#fab005", label="Plants", edgecolor="white", linewidth=0.4)
        ax.set_xlabel(label)
        ax.set_ylabel("Count")
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

        # Bottom row: top 15 nodes
        ax = axes[1, col_idx]
        top15 = metrics.nlargest(15, col)
        colors = ["#4dabf7" if t == "mine" else "#fab005" for t in top15["NODE_TYPE"]]
        names = [n[:20] + "..." if len(n) > 20 else n for n in top15["NODE"]]
        ax.barh(range(len(top15)), top15[col].values, color=colors, edgecolor="white", linewidth=0.3)
        ax.set_yticks(range(len(top15)))
        ax.set_yticklabels(names, fontsize=7)
        ax.invert_yaxis()
        ax.set_xlabel(label)
        ax.grid(True, axis="x", alpha=0.3)

    fig.suptitle("Centrality Distributions: Mines vs Plants", fontsize=13, fontweight="bold", y=0.98)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    path = os.path.join(output_dir, "Chile_Network_Centrality.png")
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")
def plot_commodity_subnetworks(sub_df, output_dir):
    """Compare commodity subnetworks by size, density, and avg distance."""
    df = sub_df[sub_df["N_EDGES"] >= 2].copy().sort_values("N_EDGES", ascending=True)

    fig, axes = plt.subplots(1, 3, figsize=(16, 6))

    # Edges
    ax = axes[0]
    colors = ["#e03131" if c == "Copper" else "#1971c2" for c in df["COMMODITY"]]
    ax.barh(df["COMMODITY"], df["N_EDGES"], color=colors, edgecolor="white", linewidth=0.3)
    ax.set_xlabel("Number of edges")
    ax.set_title("Subnetwork Size")
    ax.grid(True, axis="x", alpha=0.3)

    # Density
    ax = axes[1]
    ax.barh(df["COMMODITY"], df["DENSITY"], color="#2f9e44", edgecolor="white", linewidth=0.3)
    ax.set_xlabel("Density")
    ax.set_title("Subnetwork Density")
    ax.grid(True, axis="x", alpha=0.3)

    # Avg distance
    ax = axes[2]
    ax.barh(df["COMMODITY"], df["AVG_DISTANCE_KM"], color="#f08c00", edgecolor="white", linewidth=0.3)
    ax.set_xlabel("Avg distance (km)")
    ax.set_title("Avg Mine-Plant Distance")
    ax.grid(True, axis="x", alpha=0.3)

    fig.suptitle("Commodity Subnetwork Comparison", fontsize=13, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    path = os.path.join(output_dir, "Chile_Network_Commodity_Comparison.png")
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")
def plot_community_structure(G, metrics, partition, output_dir):
    """Community sizes, commodity breakdown, and geographic layout."""
    fig = plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)

    # --- Panel 1: Community sizes (stacked mine/plant) ---
    ax1 = fig.add_subplot(gs[0, 0])
    comm_ids = sorted(set(partition.values()), key=lambda c: -sum(1 for v in partition.values() if v == c))
    mine_counts = []
    plant_counts = []
    for cid in comm_ids:
        members = [n for n, c in partition.items() if c == cid]
        mine_counts.append(sum(1 for m in members if G.nodes[m].get("node_type") == "mine"))
        plant_counts.append(sum(1 for m in members if G.nodes[m].get("node_type") == "plant"))

    x = range(len(comm_ids))
    ax1.bar(x, mine_counts, color="#4dabf7", label="Mines", edgecolor="white", linewidth=0.3)
    ax1.bar(x, plant_counts, bottom=mine_counts, color="#fab005", label="Plants", edgecolor="white", linewidth=0.3)
    ax1.set_xticks(x)
    ax1.set_xticklabels([str(c) for c in comm_ids], fontsize=7)
    ax1.set_xlabel("Community ID")
    ax1.set_ylabel("Node count")
    ax1.set_title("Community Sizes")
    ax1.legend(fontsize=7)
    ax1.grid(True, axis="y", alpha=0.3)

    # --- Panel 2: Dominant commodity per community ---
    ax2 = fig.add_subplot(gs[0, 1])
    from collections import Counter
    comm_commodity_data = {}
    for cid in comm_ids:
        members = [n for n, c in partition.items() if c == cid]
        comms = Counter()
        for m in members:
            for nb in G.neighbors(m):
                if partition.get(nb) == cid:
                    for c in G.edges[m, nb].get("commodities", "").split(", "):
                        if c.strip():
                            comms[c.strip()] += 1
        comm_commodity_data[cid] = comms

    # Get top 6 commodities overall
    all_comms = Counter()
    for v in comm_commodity_data.values():
        all_comms.update(v)
    top_commodities = [c for c, _ in all_comms.most_common(6)]
    commodity_colors = dict(zip(top_commodities, ["#e03131", "#1971c2", "#2f9e44", "#f08c00", "#7048e8", "#0c8599"]))

    bottom = np.zeros(len(comm_ids))
    for commodity in top_commodities:
        vals = [comm_commodity_data[cid].get(commodity, 0) for cid in comm_ids]
        ax2.bar(x, vals, bottom=bottom, label=commodity, color=commodity_colors[commodity],
                edgecolor="white", linewidth=0.3)
        bottom += np.array(vals)

    ax2.set_xticks(x)
    ax2.set_xticklabels([str(c) for c in comm_ids], fontsize=7)
    ax2.set_xlabel("Community ID")
    ax2.set_ylabel("Edge count (intra-community)")
    ax2.set_title("Commodity Composition by Community")
    ax2.legend(fontsize=7, ncol=2)
    ax2.grid(True, axis="y", alpha=0.3)

    # --- Panel 3: Geographic scatter colored by community ---
    ax3 = fig.add_subplot(gs[1, :])
    cmap = plt.cm.get_cmap("tab20", max(partition.values()) + 1)
    for node in G.nodes():
        nd = G.nodes[node]
        lat, lon = nd.get("lat"), nd.get("lon")
        if lat is None or lon is None or np.isnan(lat) or np.isnan(lon):
            continue
        cid = partition[node]
        marker = "o" if nd.get("node_type") == "mine" else "s"
        size = 15 + 3 * G.degree(node)
        ax3.scatter(lon, lat, c=[cmap(cid)], s=size, marker=marker, alpha=0.7,
                    edgecolors="white", linewidths=0.3, zorder=2)

    ax3.set_xlabel("Longitude")
    ax3.set_ylabel("Latitude")
    ax3.set_title("Facilities by Community (circle=mine, square=plant)")
    ax3.grid(True, alpha=0.3)
    ax3.set_aspect("auto")

    fig.suptitle("Community Structure", fontsize=13, fontweight="bold", y=0.99)
    path = os.path.join(output_dir, "Chile_Network_Communities.png")
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")

def plot_community_structure(G, metrics, partition, output_dir):
    """Community sizes, commodity breakdown, and geographic layout."""
    fig = plt.figure(figsize=(16, 10))
    gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3)

    # --- Panel 1: Community sizes (stacked mine/plant) ---
    ax1 = fig.add_subplot(gs[0, 0])
    comm_ids = sorted(set(partition.values()), key=lambda c: -sum(1 for v in partition.values() if v == c))
    mine_counts = []
    plant_counts = []
    for cid in comm_ids:
        members = [n for n, c in partition.items() if c == cid]
        mine_counts.append(sum(1 for m in members if G.nodes[m].get("node_type") == "mine"))
        plant_counts.append(sum(1 for m in members if G.nodes[m].get("node_type") == "plant"))

    x = range(len(comm_ids))
    ax1.bar(x, mine_counts, color="#4dabf7", label="Mines", edgecolor="white", linewidth=0.3)
    ax1.bar(x, plant_counts, bottom=mine_counts, color="#fab005", label="Plants", edgecolor="white", linewidth=0.3)
    ax1.set_xticks(x)
    ax1.set_xticklabels([str(c) for c in comm_ids], fontsize=7)
    ax1.set_xlabel("Community ID")
    ax1.set_ylabel("Node count")
    ax1.set_title("Community Sizes")
    ax1.legend(fontsize=7)
    ax1.grid(True, axis="y", alpha=0.3)

    # --- Panel 2: Dominant commodity per community ---
    ax2 = fig.add_subplot(gs[0, 1])
    from collections import Counter
    comm_commodity_data = {}
    for cid in comm_ids:
        members = [n for n, c in partition.items() if c == cid]
        comms = Counter()
        for m in members:
            for nb in G.neighbors(m):
                if partition.get(nb) == cid:
                    for c in G.edges[m, nb].get("commodities", "").split(", "):
                        if c.strip():
                            comms[c.strip()] += 1
        comm_commodity_data[cid] = comms

    # Get top 6 commodities overall
    all_comms = Counter()
    for v in comm_commodity_data.values():
        all_comms.update(v)
    top_commodities = [c for c, _ in all_comms.most_common(6)]
    commodity_colors = dict(zip(top_commodities, ["#e03131", "#1971c2", "#2f9e44", "#f08c00", "#7048e8", "#0c8599"]))

    bottom = np.zeros(len(comm_ids))
    for commodity in top_commodities:
        vals = [comm_commodity_data[cid].get(commodity, 0) for cid in comm_ids]
        ax2.bar(x, vals, bottom=bottom, label=commodity, color=commodity_colors[commodity],
                edgecolor="white", linewidth=0.3)
        bottom += np.array(vals)

    ax2.set_xticks(x)
    ax2.set_xticklabels([str(c) for c in comm_ids], fontsize=7)
    ax2.set_xlabel("Community ID")
    ax2.set_ylabel("Edge count (intra-community)")
    ax2.set_title("Commodity Composition by Community")
    ax2.legend(fontsize=7, ncol=2)
    ax2.grid(True, axis="y", alpha=0.3)

    # --- Panel 3: Geographic scatter colored by community ---
    ax3 = fig.add_subplot(gs[1, :])
    cmap = plt.cm.get_cmap("tab20", max(partition.values()) + 1)
    for node in G.nodes():
        nd = G.nodes[node]
        lat, lon = nd.get("lat"), nd.get("lon")
        if lat is None or lon is None or np.isnan(lat) or np.isnan(lon):
            continue
        cid = partition[node]
        marker = "o" if nd.get("node_type") == "mine" else "s"
        size = 15 + 3 * G.degree(node)
        ax3.scatter(lon, lat, c=[cmap(cid)], s=size, marker=marker, alpha=0.7,
                    edgecolors="white", linewidths=0.3, zorder=2)

    ax3.set_xlabel("Longitude")
    ax3.set_ylabel("Latitude")
    ax3.set_title("Facilities by Community (circle=mine, square=plant)")
    ax3.grid(True, alpha=0.3)
    ax3.set_aspect("auto")

    fig.suptitle("Community Structure", fontsize=13, fontweight="bold", y=0.99)
    path = os.path.join(output_dir, "Chile_Network_Communities.png")
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")
def plot_projections(mine_proj, plant_proj, output_dir):
    """Degree distributions of the two unipartite projections."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for ax, proj, title, color in [
        (axes[0], mine_proj, "Mine-Mine Projection\n(shared downstream plants)", "#4dabf7"),
        (axes[1], plant_proj, "Plant-Plant Projection\n(shared upstream mines)", "#fab005"),
    ]:
        if proj.number_of_edges() == 0:
            ax.text(0.5, 0.5, "No edges", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(title)
            continue

        degs = [d for _, d in proj.degree()]
        ax.hist(degs, bins=25, color=color, edgecolor="white", linewidth=0.4, alpha=0.8)
        ax.axvline(np.mean(degs), color="#333", linestyle="--", linewidth=1, label=f"mean={np.mean(degs):.1f}")
        ax.axvline(np.median(degs), color="#666", linestyle=":", linewidth=1, label=f"median={np.median(degs):.1f}")
        ax.set_xlabel("Degree")
        ax.set_ylabel("Count")
        ax.set_title(title)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

    fig.suptitle("Projected Network Degree Distributions", fontsize=13, fontweight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    path = os.path.join(output_dir, "Chile_Network_Projections.png")
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def generate_all_plots(G, metrics, partition, links, sub_df, mine_proj, plant_proj, output_dir):
    print("\n" + "=" * 60)
    print("Generating additional visualizations...")
    print("=" * 60)
    plot_centrality_distributions(metrics, output_dir)
    plot_community_structure(G, metrics, partition, output_dir)
    plot_commodity_subnetworks(sub_df, output_dir)
    plot_projections(mine_proj, plant_proj, output_dir)

generate_all_plots(
    results["G"], results["metrics"], results["partition"],
    results["links"], results["sub_df"],
    results["mine_proj"], results["plant_proj"],
    OUTPUT_DIR
)


Generating additional visualizations...
  Saved: /Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile/Chile_Network_Centrality.png
  Saved: /Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile/Chile_Network_Communities.png
  Saved: /Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile/Chile_Network_Commodity_Comparison.png
  Saved: /Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile/Chile_Network_Projections.png


In [79]:
"""
Geographic mapping of Chile mine-to-processing-plant supply chain links.

Reads Chile_Mine_Plant_Links.csv and produces:
  1. Interactive Leaflet/Folium HTML map with commodity filtering
  2. Static matplotlib figure suitable for reports

Inputs:
  - Chile_Mine_Plant_Links.csv (same as network analysis script)

Outputs:
  - Chile_Supply_Chain_Map.html  (interactive)
  - Chile_Supply_Chain_Map.png   (static, report-ready)
"""

import os
import pandas as pd
import numpy as np
import folium
from folium import plugins
from collections import defaultdict, Counter
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = "/Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile"
LINKS_PATH = os.path.join(BASE_DIR, "Chile_Mine_Plant_Links.csv")
OUTPUT_DIR = BASE_DIR

# Commodity color palette (consistent across both maps)
COMMODITY_COLORS = {
    "Copper":      "#d63031",
    "Gold":        "#fdcb6e",
    "Silver":      "#636e72",
    "Iron":        "#e17055",
    "Molybdenum":  "#6c5ce7",
    "Iodine":      "#00b894",
    "Nitrate":     "#00cec9",
    "Rhenium":     "#a29bfe",
    "Manganese":   "#e84393",
    "Zinc":        "#74b9ff",
    "Boron":       "#55efc4",
    "Lithium":     "#fd79a8",
}
DEFAULT_COLOR = "#b2bec3"


def get_commodity_color(commodity):
    return COMMODITY_COLORS.get(commodity, DEFAULT_COLOR)


def get_primary_commodity(shared_commodities_str):
    """Return the first listed commodity as 'primary'."""
    parts = [c.strip() for c in str(shared_commodities_str).split(",") if c.strip()]
    return parts[0] if parts else "Unknown"


# ============================================================
# LOAD DATA
# ============================================================

def load_links(path):
    df = pd.read_csv(path)
    # Drop rows missing coordinates
    df = df.dropna(subset=["MINE_LAT", "MINE_LON", "PLANT_LAT", "PLANT_LON"])
    df["PRIMARY_COMMODITY"] = df["SHARED_COMMODITIES"].apply(get_primary_commodity)
    print(f"Loaded {len(df)} links ({df['MINE_NAME'].nunique()} mines, {df['PLANT_NAME'].nunique()} plants)")
    return df


# ============================================================
# 1. INTERACTIVE FOLIUM MAP
# ============================================================

def build_interactive_map(df, output_dir):
    print("Building interactive supply chain map...")

    # Center on Chile
    center_lat = df[["MINE_LAT", "PLANT_LAT"]].values.flatten().mean()
    center_lon = df[["MINE_LON", "PLANT_LON"]].values.flatten().mean()

    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=6,
        tiles=None,
        control_scale=True,
    )

    # Tile layers
    folium.TileLayer(
        "cartodbpositron", name="Light basemap", control=True
    ).add_to(m)
    folium.TileLayer(
        "cartodbdark_matter", name="Dark basemap", control=True
    ).add_to(m)

    # --- Aggregate unique mines and plants with degree info ---
    mine_degree = df.groupby("MINE_NAME").size().to_dict()
    plant_degree = df.groupby("PLANT_NAME").size().to_dict()

    mines = df.drop_duplicates("MINE_NAME").set_index("MINE_NAME")
    plants = df.drop_duplicates("PLANT_NAME").set_index("PLANT_NAME")

    # --- Create feature groups per commodity ---
    commodities = sorted(df["PRIMARY_COMMODITY"].unique())

    # All-links layer (thin, grey, always visible for context)
    all_links_group = folium.FeatureGroup(name="All supply links", show=True)
    for _, row in df.iterrows():
        folium.PolyLine(
            locations=[
                [row["MINE_LAT"], row["MINE_LON"]],
                [row["PLANT_LAT"], row["PLANT_LON"]],
            ],
            color=get_commodity_color(row["PRIMARY_COMMODITY"]),
            weight=1.2,
            opacity=0.25,
            tooltip=f"{row['MINE_NAME']} → {row['PLANT_NAME']}<br>"
                    f"{row['SHARED_COMMODITIES']}<br>"
                    f"{row['DISTANCE_KM']:.1f} km",
        ).add_to(all_links_group)
    all_links_group.add_to(m)

    # Per-commodity link layers
    for commodity in commodities:
        sub = df[df["PRIMARY_COMMODITY"] == commodity]
        if len(sub) == 0:
            continue
        fg = folium.FeatureGroup(name=f"{commodity} links ({len(sub)})", show=False)
        color = get_commodity_color(commodity)
        for _, row in sub.iterrows():
            folium.PolyLine(
                locations=[
                    [row["MINE_LAT"], row["MINE_LON"]],
                    [row["PLANT_LAT"], row["PLANT_LON"]],
                ],
                color=color,
                weight=2.0,
                opacity=0.6,
                tooltip=f"{row['MINE_NAME']} → {row['PLANT_NAME']}<br>"
                        f"{row['SHARED_COMMODITIES']}<br>"
                        f"{row['DISTANCE_KM']:.1f} km",
            ).add_to(fg)
        fg.add_to(m)

    # --- Mine markers ---
    mine_group = folium.FeatureGroup(name="Mines", show=True)
    for name, row in mines.iterrows():
        deg = mine_degree.get(name, 1)
        radius = 3 + min(deg * 0.8, 12)
        # Collect all commodities this mine handles
        mine_comms = df[df["MINE_NAME"] == name]["SHARED_COMMODITIES"].str.cat(sep=", ")
        unique_comms = sorted(set(c.strip() for c in mine_comms.split(",") if c.strip()))

        popup_html = (
            f"<b>{name}</b><br>"
            f"Type: {row.get('MINE_TYPE', 'Mine')}<br>"
            f"Status: {row.get('MINE_STATUS', '')}<br>"
            f"Region: {row.get('MINE_REGION', '')}<br>"
            f"Operator: {row.get('MINE_OPERATOR', '')}<br>"
            f"Connections: {deg}<br>"
            f"Commodities: {', '.join(unique_comms)}"
        )

        folium.CircleMarker(
            location=[row["MINE_LAT"], row["MINE_LON"]],
            radius=radius,
            color="#2d3436",
            fill=True,
            fill_color="#4dabf7",
            fill_opacity=0.75,
            weight=0.6,
            popup=folium.Popup(popup_html, max_width=280),
            tooltip=f"{name} (mine, deg={deg})",
        ).add_to(mine_group)
    mine_group.add_to(m)

    # --- Plant markers ---
    plant_group = folium.FeatureGroup(name="Plants", show=True)
    for name, row in plants.iterrows():
        deg = plant_degree.get(name, 1)
        radius = 4 + min(deg * 0.8, 14)
        plant_comms = df[df["PLANT_NAME"] == name]["SHARED_COMMODITIES"].str.cat(sep=", ")
        unique_comms = sorted(set(c.strip() for c in plant_comms.split(",") if c.strip()))

        popup_html = (
            f"<b>{name}</b><br>"
            f"Type: {row.get('PLANT_TYPE', 'Plant')}<br>"
            f"Status: {row.get('PLANT_STATUS', '')}<br>"
            f"Operator: {row.get('PLANT_OPERATOR', '')}<br>"
            f"Connections: {deg}<br>"
            f"Commodities: {', '.join(unique_comms)}"
        )
        cap = row.get("PLANT_CAPACITY")
        if pd.notna(cap):
            popup_html += f"<br>Capacity: {cap} {row.get('PLANT_CAPACITY_UNITS', '')}"

        folium.RegularPolygonMarker(
            location=[row["PLANT_LAT"], row["PLANT_LON"]],
            number_of_sides=4,
            radius=radius,
            color="#2d3436",
            fill=True,
            fill_color="#fab005",
            fill_opacity=0.80,
            weight=0.6,
            rotation=45,
            popup=folium.Popup(popup_html, max_width=280),
            tooltip=f"{name} (plant, deg={deg})",
        ).add_to(plant_group)
    plant_group.add_to(m)

    # Layer control
    folium.LayerControl(collapsed=False).add_to(m)

    # Legend (manual HTML)
    legend_html = """
    <div style="position:fixed; bottom:30px; left:30px; z-index:9999;
                background:white; padding:12px 16px; border-radius:6px;
                box-shadow:0 2px 8px rgba(0,0,0,0.2); font-size:12px;
                font-family:sans-serif; line-height:1.6;">
      <b>Legend</b><br>
      <span style="display:inline-block;width:12px;height:12px;
            border-radius:50%;background:#4dabf7;margin-right:6px;
            vertical-align:middle;border:1px solid #2d3436;"></span> Mine<br>
      <span style="display:inline-block;width:12px;height:12px;
            background:#fab005;margin-right:6px;
            vertical-align:middle;border:1px solid #2d3436;
            transform:rotate(45deg);"></span> Processing plant<br>
      <hr style="margin:4px 0;">
    """
    for comm in commodities:
        c = get_commodity_color(comm)
        legend_html += (
            f'<span style="display:inline-block;width:20px;height:3px;'
            f'background:{c};margin-right:6px;vertical-align:middle;"></span>'
            f'{comm}<br>'
        )
    legend_html += "</div>"
    m.get_root().html.add_child(folium.Element(legend_html))

    path = os.path.join(output_dir, "Chile_Supply_Chain_Map.html")
    m.save(path)
    print(f"  Saved: {path}")
    return path


# ============================================================
# 2. STATIC MATPLOTLIB MAP
# ============================================================

def build_static_map(df, output_dir):
    print("Building static supply chain map...")

    fig, ax = plt.subplots(figsize=(10, 18))

    # --- Draw links, colored by primary commodity ---
    commodities = sorted(df["PRIMARY_COMMODITY"].unique())

    for _, row in df.iterrows():
        color = get_commodity_color(row["PRIMARY_COMMODITY"])
        ax.plot(
            [row["MINE_LON"], row["PLANT_LON"]],
            [row["MINE_LAT"], row["PLANT_LAT"]],
            color=color, alpha=0.12, linewidth=0.5, zorder=1,
        )

    # --- Aggregate mine/plant positions and degrees ---
    mine_degree = df.groupby("MINE_NAME").size()
    plant_degree = df.groupby("PLANT_NAME").size()

    mines = df.drop_duplicates("MINE_NAME").set_index("MINE_NAME")
    plants = df.drop_duplicates("PLANT_NAME").set_index("PLANT_NAME")

    max_deg = max(mine_degree.max(), plant_degree.max())

    # Mines
    for name, row in mines.iterrows():
        deg = mine_degree.get(name, 1)
        size = 8 + 60 * (deg / max_deg)
        ax.scatter(
            row["MINE_LON"], row["MINE_LAT"],
            s=size, c="#4dabf7", marker="o",
            edgecolors="white", linewidths=0.3,
            alpha=0.75, zorder=3,
        )

    # Plants
    for name, row in plants.iterrows():
        deg = plant_degree.get(name, 1)
        size = 12 + 80 * (deg / max_deg)
        ax.scatter(
            row["PLANT_LON"], row["PLANT_LAT"],
            s=size, c="#fab005", marker="s",
            edgecolors="#333333", linewidths=0.4,
            alpha=0.85, zorder=4,
        )

    # Label top-degree facilities
    top_mines = mine_degree.nlargest(8).index
    top_plants = plant_degree.nlargest(8).index

    for name in top_mines:
        if name in mines.index:
            row = mines.loc[name]
            label = name[:25] + "..." if len(name) > 25 else name
            ax.annotate(
                label,
                (row["MINE_LON"], row["MINE_LAT"]),
                fontsize=5, color="#2d3436", fontweight="bold",
                xytext=(4, 3), textcoords="offset points",
                zorder=5,
            )

    for name in top_plants:
        if name in plants.index:
            row = plants.loc[name]
            label = name[:25] + "..." if len(name) > 25 else name
            ax.annotate(
                label,
                (row["PLANT_LON"], row["PLANT_LAT"]),
                fontsize=5, color="#6c3c00", fontweight="bold",
                xytext=(4, -5), textcoords="offset points",
                zorder=5,
            )

    # --- Legend ---
    handles = [
        mlines.Line2D([], [], marker="o", color="w", markerfacecolor="#4dabf7",
                       markeredgecolor="white", markersize=7, label="Mine"),
        mlines.Line2D([], [], marker="s", color="w", markerfacecolor="#fab005",
                       markeredgecolor="#333", markersize=7, label="Processing plant"),
    ]
    for comm in commodities:
        handles.append(
            mlines.Line2D([], [], color=get_commodity_color(comm),
                          linewidth=2, label=comm)
        )
    ax.legend(handles=handles, loc="lower left", fontsize=6, framealpha=0.9,
              edgecolor="#cccccc")

    ax.set_xlabel("Longitude", fontsize=9)
    ax.set_ylabel("Latitude", fontsize=9)
    ax.set_title(
        f"Chile Mine-to-Plant Supply Chain\n"
        f"{df['MINE_NAME'].nunique()} mines, {df['PLANT_NAME'].nunique()} plants, "
        f"{len(df)} links",
        fontsize=11, fontweight="bold",
    )
    ax.set_facecolor("#fafafa")
    ax.grid(True, alpha=0.2, linewidth=0.4)

    # Pad extent slightly
    lon_vals = pd.concat([df["MINE_LON"], df["PLANT_LON"]])
    lat_vals = pd.concat([df["MINE_LAT"], df["PLANT_LAT"]])
    ax.set_xlim(lon_vals.min() - 0.5, lon_vals.max() + 0.5)
    ax.set_ylim(lat_vals.min() - 0.5, lat_vals.max() + 0.5)

    fig.tight_layout()
    path = os.path.join(output_dir, "Chile_Supply_Chain_Map.png")
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  Saved: {path}")
    return path


# ============================================================
# MAIN
# ============================================================

def main():
    df = load_links(LINKS_PATH)
    build_interactive_map(df, OUTPUT_DIR)
    build_static_map(df, OUTPUT_DIR)
    print("\nDone.")

if __name__ == "__main__":
    main()

Loaded 1370 links (206 mines, 78 plants)
Building interactive supply chain map...
  Saved: /Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile/Chile_Supply_Chain_Map.html
Building static supply chain map...
  Saved: /Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile/Chile_Supply_Chain_Map.png

Done.


## Later

In [ ]:
"""
Chile Minerals Supply Chain Map
- Groups mines with their processing plants (by operator + commodity + proximity)
- Shows international destinations for Chilean mineral exports
- Draws flow lines from mines -> domestic plants -> ports -> international smelters/refineries

"""

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from math import radians, cos, sin, asin, sqrt
import pyEOF

# ============================================================
# 1. LOAD DATA
# ============================================================

# Update this path to your local file
df = pd.read_csv("Chile_Minerals_Supply_Chain.csv")
print(f"Loaded {len(df)} records")

# ============================================================
# 2. CLASSIFY FACILITIES
# ============================================================

mines = df[df["FACILITY_TYPE"].isin(["Mine (active)", "Mine (idle)", "Prospect/Project"])].copy()
plants = df[~df["FACILITY_TYPE"].isin(["Mine (active)", "Mine (idle)", "Prospect/Project"])].copy()

print(f"Mines/deposits: {len(mines)}")
print(f"Processing facilities: {len(plants)}")

# ============================================================
# 3. MAJOR CHILEAN EXPORT PORTS (manually added)
# ============================================================

ports = pd.DataFrame([
    {"name": "Antofagasta Port", "lat": -23.6345, "lng": -70.3980, "commodity": "Copper, Lithium", "note": "Main copper concentrate export port"},
    {"name": "Mejillones Port", "lat": -23.1000, "lng": -70.4500, "commodity": "Copper", "note": "BHP/Escondida concentrate terminal"},
    {"name": "San Antonio Port", "lat": -33.5950, "lng": -71.6210, "commodity": "Copper", "note": "Central Chile copper exports"},
    {"name": "Valparaíso Port", "lat": -33.0472, "lng": -71.6127, "commodity": "Copper, General", "note": "General cargo and copper"},
    {"name": "Barquito Port", "lat": -27.0800, "lng": -70.8400, "commodity": "Iron, Copper", "note": "CMP iron ore + copper concentrate"},
    {"name": "Guacolda II Port", "lat": -28.4700, "lng": -71.2300, "commodity": "Iron", "note": "CMP iron ore pellets"},
    {"name": "Patillos Port", "lat": -20.7500, "lng": -70.1900, "commodity": "Salt, Nitrates, Iodine", "note": "SQM/K+S exports"},
    {"name": "Tocopilla Port", "lat": -22.0960, "lng": -70.1980, "commodity": "Copper", "note": "Copper exports"},
    {"name": "Iquique Port", "lat": -20.2133, "lng": -70.1503, "commodity": "Nitrates, Iodine, Lithium", "note": "Northern exports"},
    {"name": "Arica Port", "lat": -18.4746, "lng": -70.3120, "commodity": "Boron", "note": "Boron and mineral exports"},
])

# ============================================================
# 4. INTERNATIONAL PROCESSING DESTINATIONS
#    Based on 2024 trade data (WITS/Comtrade)
# ============================================================

intl_destinations = pd.DataFrame([
    # Copper concentrate destinations (2024 data)
    {"name": "China Smelters", "lat": 31.2, "lng": 121.5, "commodity": "Copper",
     "note": "67% of Chilean Cu concentrate ($21B). Yangshan, Tongling, Guixi smelters",
     "share": 0.67, "value_usd_b": 21.0},
    {"name": "Japan Smelters", "lat": 34.7, "lng": 135.5, "commodity": "Copper",
     "note": "18% of Chilean Cu concentrate ($5.6B). Saganoseki, Onahama, Tamano",
     "share": 0.18, "value_usd_b": 5.6},
    {"name": "India Smelters", "lat": 19.1, "lng": 72.9, "commodity": "Copper",
     "note": "5% of Chilean Cu concentrate ($1.4B). Tuticorin, Dahej",
     "share": 0.05, "value_usd_b": 1.4},
    {"name": "South Korea Smelters", "lat": 35.2, "lng": 129.1, "commodity": "Copper",
     "note": "4% of Chilean Cu concentrate ($1.1B). Onsan refinery",
     "share": 0.04, "value_usd_b": 1.1},
    {"name": "Germany Smelters", "lat": 51.3, "lng": 9.5, "commodity": "Copper",
     "note": "~1% of Chilean Cu concentrate. Aurubis Hamburg",
     "share": 0.01, "value_usd_b": 0.4},

    # Lithium destinations
    {"name": "China Li Processing", "lat": 29.4, "lng": 106.5, "commodity": "Lithium",
     "note": "~70% of global Li refining. Ganfeng, Tianqi (SQM partner)",
     "share": 0.70, "value_usd_b": 3.5},
    {"name": "South Korea Li Processing", "lat": 36.5, "lng": 127.0, "commodity": "Lithium",
     "note": "Battery-grade Li for LG, Samsung SDI, SK",
     "share": 0.15, "value_usd_b": 0.8},
    {"name": "Japan Li Processing", "lat": 35.7, "lng": 139.7, "commodity": "Lithium",
     "note": "Panasonic, Sumitomo Metal Mining supply chain",
     "share": 0.10, "value_usd_b": 0.5},

    # Iron ore
    {"name": "China Steel Mills", "lat": 39.9, "lng": 116.4, "commodity": "Iron",
     "note": "Primary destination for CMP iron ore pellets",
     "share": 0.60, "value_usd_b": 1.2},

    # Iodine/Nitrates
    {"name": "USA Chemical Plants", "lat": 29.7, "lng": -95.4, "commodity": "Iodine",
     "note": "SQM iodine exports to US pharmaceutical/industrial",
     "share": 0.25, "value_usd_b": 0.3},
])

# ============================================================
# 5. LINK MINES TO PLANTS (by operator + commodity + proximity)
# ============================================================

def haversine(lat1, lon1, lat2, lon2):
    """Distance in km between two points."""
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 2 * 6371 * asin(sqrt(a))

links = []  # mine -> plant connections

for _, plant in plants.iterrows():
    if pd.isna(plant["LATITUD"]) or pd.isna(plant["LONGITUD"]):
        continue

    plant_op = str(plant.get("OPERATOR_NAME", "")).lower().strip()
    plant_comm = str(plant.get("PRIMARY_COMMODITY", "")).lower().strip()
    plant_lat = plant["LATITUD"]
    plant_lng = plant["LONGITUD"]

    best_mine = None
    best_dist = 999999

    for _, mine in mines.iterrows():
        if pd.isna(mine["LATITUD"]) or pd.isna(mine["LONGITUD"]):
            continue

        mine_comm = str(mine.get("PRIMARY_COMMODITY", "")).lower().strip()
        mine_op = str(mine.get("OPERATOR_NAME", "")).lower().strip()

        # Must share a commodity
        if mine_comm != plant_comm and plant_comm not in str(mine.get("ALL_COMMODITIES", "")).lower():
            continue

        dist = haversine(mine["LATITUD"], mine["LONGITUD"], plant_lat, plant_lng)

        # Match by operator name (fuzzy) or by proximity (<50km)
        op_match = False
        if plant_op and mine_op and plant_op != "nan" and mine_op != "nan":
            # Check if key words overlap
            plant_words = set(plant_op.split())
            mine_words = set(mine_op.split())
            if len(plant_words & mine_words) >= 2:
                op_match = True

        if op_match and dist < 200:
            if dist < best_dist:
                best_dist = dist
                best_mine = mine
        elif dist < 50:
            if dist < best_dist:
                best_dist = dist
                best_mine = mine

    if best_mine is not None:
        links.append({
            "mine_name": best_mine["FACILITY_NAME"],
            "mine_lat": best_mine["LATITUD"],
            "mine_lng": best_mine["LONGITUD"],
            "plant_name": plant["FACILITY_NAME"],
            "plant_lat": plant_lat,
            "plant_lng": plant_lng,
            "commodity": plant["PRIMARY_COMMODITY"],
            "distance_km": round(best_dist, 1),
            "link_type": "mine_to_plant"
        })

print(f"Mine-to-plant links found: {len(links)}")

# ============================================================
# 6. BUILD THE MAP
# ============================================================

fig = go.Figure()

# --- Color scheme ---
COMM_COLORS = {
    "Copper": "#e07020", "Gold": "#ffd700", "Silver": "#c0c0c0",
    "Iron": "#8b0000", "Lithium": "#00ccff", "Nitrate": "#90ee90",
    "Boron": "#ff69b4", "Manganese": "#9370db", "Zinc": "#20b2aa",
    "Titanium": "#ff4500", "Rare Earths": "#ff00ff", "Molybdenum": "#6a5acd",
    "Cobalt": "#1e90ff", "Rhenium": "#daa520", "Iodine": "#98fb98",
    "Cement": "#808080", "Calcium Carbonate": "#d2b48c", "Silica": "#f5f5dc",
    "Sulfuric Acid": "#ffff00", "Pumicite": "#bc8f8f",
}

def get_color(comm):
    return COMM_COLORS.get(comm, "#999999")

# --- A. Mine-to-plant connection lines ---
for link in links:
    fig.add_trace(go.Scattermap(
        lat=[link["mine_lat"], link["plant_lat"]],
        lon=[link["mine_lng"], link["plant_lng"]],
        mode="lines",
        line=dict(width=1, color="rgba(255,255,255,0.15)"),
        hoverinfo="text",
        text=f"{link['mine_name']} -> {link['plant_name']} ({link['commodity']}, {link['distance_km']}km)",
        showlegend=False,
    ))

# --- B. Export flow lines (ports -> international) ---
# Connect main ports to international destinations
port_coords = {"Copper": (-23.6345, -70.3980), "Lithium": (-23.6345, -70.3980),
               "Iron": (-28.4700, -71.2300), "Iodine": (-20.2133, -70.1503)}

for _, dest in intl_destinations.iterrows():
    origin = port_coords.get(dest["commodity"], (-23.6345, -70.3980))
    width = max(1, dest["share"] * 8)
    color = get_color(dest["commodity"])

    fig.add_trace(go.Scattermap(
        lat=[origin[0], dest["lat"]],
        lon=[origin[1], dest["lng"]],
        mode="lines",
        line=dict(width=width, color=color),
        opacity=0.4,
        hoverinfo="text",
        text=f"{dest['commodity']}: Chile -> {dest['name']}<br>{dest['share']*100:.0f}% share (${dest['value_usd_b']:.1f}B)<br>{dest['note']}",
        showlegend=False,
    ))

# --- C. Mines (active) ---
active = mines[mines["FACILITY_TYPE"] == "Mine (active)"].dropna(subset=["LATITUD", "LONGITUD"])
fig.add_trace(go.Scattermap(
    lat=active["LATITUD"], lon=active["LONGITUD"],
    mode="markers",
    marker=dict(
        size=8,
        color=[get_color(c) for c in active["PRIMARY_COMMODITY"]],
        opacity=0.9,
    ),
    text=active.apply(lambda r: f"<b>{r['FACILITY_NAME']}</b><br>Active Mine<br>{r['ALL_COMMODITIES']}<br>{r['REGION']}", axis=1),
    hoverinfo="text",
    name="Active Mines",
))

# --- D. Mines (idle) ---
idle = mines[mines["FACILITY_TYPE"] == "Mine (idle)"].dropna(subset=["LATITUD", "LONGITUD"])
fig.add_trace(go.Scattermap(
    lat=idle["LATITUD"], lon=idle["LONGITUD"],
    mode="markers",
    marker=dict(
        size=5,
        color=[get_color(c) for c in idle["PRIMARY_COMMODITY"]],
        opacity=0.4,
    ),
    text=idle.apply(lambda r: f"<b>{r['FACILITY_NAME']}</b><br>Idle Mine<br>{r['ALL_COMMODITIES']}<br>{r['REGION']}", axis=1),
    hoverinfo="text",
    name="Idle Mines",
))

# --- E. Prospects ---
prosp = mines[mines["FACILITY_TYPE"] == "Prospect/Project"].dropna(subset=["LATITUD", "LONGITUD"])
fig.add_trace(go.Scattermap(
    lat=prosp["LATITUD"], lon=prosp["LONGITUD"],
    mode="markers",
    marker=dict(
        size=4,
        color="gray",
        opacity=0.3,
    ),
    text=prosp.apply(lambda r: f"<b>{r['FACILITY_NAME']}</b><br>Prospect/Project<br>{r['ALL_COMMODITIES']}<br>{r['REGION']}", axis=1),
    hoverinfo="text",
    name="Prospects",
))

# --- F. Processing plants ---
# Separate smelters/refineries from other plants
smelters = plants[plants["FACILITY_TYPE"].isin(["Smelter", "Refinery"])].dropna(subset=["LATITUD", "LONGITUD"])
other_plants = plants[~plants["FACILITY_TYPE"].isin(["Smelter", "Refinery"])].dropna(subset=["LATITUD", "LONGITUD"])

fig.add_trace(go.Scattermap(
    lat=smelters["LATITUD"], lon=smelters["LONGITUD"],
    mode="markers",
    marker=dict(
        size=14,
        color=[get_color(c) for c in smelters["PRIMARY_COMMODITY"]],
        symbol="square",
        opacity=0.95,
    ),
    text=smelters.apply(lambda r: f"<b>{r['FACILITY_NAME']}</b><br>{r['FACILITY_TYPE']}<br>{r['PRIMARY_COMMODITY']}<br>Operator: {r['OPERATOR_NAME']}<br>Capacity: {r['CAPACITY']} {r['CAPACITY_UNITS']}", axis=1),
    hoverinfo="text",
    name="Smelters/Refineries",
))

fig.add_trace(go.Scattermap(
    lat=other_plants["LATITUD"], lon=other_plants["LONGITUD"],
    mode="markers",
    marker=dict(
        size=8,
        color=[get_color(c) for c in other_plants["PRIMARY_COMMODITY"]],
        symbol="diamond",
        opacity=0.7,
    ),
    text=other_plants.apply(lambda r: f"<b>{r['FACILITY_NAME']}</b><br>{r['FACILITY_TYPE']}<br>{r['PRIMARY_COMMODITY']}<br>Operator: {r['OPERATOR_NAME']}<br>Capacity: {r['CAPACITY']} {r['CAPACITY_UNITS']}", axis=1),
    hoverinfo="text",
    name="Processing Plants",
))

# --- G. Ports ---
fig.add_trace(go.Scattermap(
    lat=ports["lat"], lon=ports["lng"],
    mode="markers+text",
    marker=dict(size=10, color="white", symbol="harbor", opacity=0.9),
    text=ports["name"],
    textposition="top right",
    textfont=dict(size=8, color="white"),
    hovertext=ports.apply(lambda r: f"<b>{r['name']}</b><br>{r['commodity']}<br>{r['note']}", axis=1),
    hoverinfo="text",
    name="Export Ports",
))

# --- H. International destinations ---
fig.add_trace(go.Scattermap(
    lat=intl_destinations["lat"], lon=intl_destinations["lng"],
    mode="markers+text",
    marker=dict(
        size=intl_destinations["value_usd_b"] * 3 + 6,
        color=[get_color(c) for c in intl_destinations["commodity"]],
        symbol="star",
        opacity=0.9,
    ),
    text=intl_destinations["name"],
    textposition="top right",
    textfont=dict(size=9, color="white"),
    hovertext=intl_destinations.apply(
        lambda r: f"<b>{r['name']}</b><br>{r['commodity']}<br>{r['share']*100:.0f}% of Chilean exports<br>${r['value_usd_b']:.1f}B (2024)<br>{r['note']}", axis=1),
    hoverinfo="text",
    name="International Processors",
))

# ============================================================
# 7. LAYOUT
# ============================================================

fig.update_layout(
    title=dict(
        text="Chile Critical Minerals Supply Chain<br><sup>Mines (Sernageomin 2025) + Processing Plants (USGS) + Export Flows (2024 trade data)</sup>",
        font=dict(size=18, color="white"),
    ),
    map=dict(
        style="carto-darkmatter",
        center=dict(lat=-10, lon=-20),
        zoom=1.8,
    ),
    legend=dict(
        bgcolor="rgba(20,20,20,0.85)",
        font=dict(color="white", size=11),
        x=0.01, y=0.99,
    ),
    paper_bgcolor="#111",
    width=1400,
    height=900,
    margin=dict(l=0, r=0, t=60, b=0),
)

# ============================================================
# 8. SAVE
# ============================================================

fig.write_html("chile_supply_chain_map.html", include_plotlyjs=True)
print("Saved: chile_supply_chain_map.html")

# Also save the mine-plant links as CSV
links_df = pd.DataFrame(links)
links_df.to_csv("mine_plant_links.csv", index=False)
print(f"Saved: mine_plant_links.csv ({len(links_df)} links)")

fig.show()
pyEOF

Loaded 461 records
Mines/deposits: 267
Processing facilities: 194
Mine-to-plant links found: 82
Saved: chile_supply_chain_map.html
Saved: mine_plant_links.csv (82 links)


<module 'pyEOF' from '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pyEOF/__init__.py'>